In [1]:
import numpy as np
import os
import sys
import joblib
from datetime import datetime

try:
    import tensorflow as tf
    TF_VERSION = tuple(int(x) for x in tf.__version__.split(".")[:2])
    MODEL_EXT  = ".keras" if TF_VERSION >= (2, 16) else ".h5"
except ImportError:
    print("ERROR: pip install tensorflow")
    sys.exit(1)

from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, accuracy_score, f1_score
)


DATA_DIR  = "data"
MODEL_DIR = "models"
TIMESTEPS = joblib.load(os.path.join(DATA_DIR, "lstm_timesteps.save")) \
            if os.path.exists(os.path.join(DATA_DIR, "lstm_timesteps.save")) else 10

X_test = np.load(os.path.join(DATA_DIR, "X_test.npy")).astype(np.float32)
y_test = np.load(os.path.join(DATA_DIR, "y_test.npy")).astype(np.int32)

def get_metrics(y_true, y_pred, y_prob=None):
    r = classification_report(y_true, y_pred, output_dict=True, zero_division=0)
    return {
        "Accuracy"         : accuracy_score(y_true, y_pred),
        "Macro F1"         : f1_score(y_true, y_pred, average="macro", zero_division=0),
        "ROC-AUC"          : roc_auc_score(y_true, y_prob) if y_prob is not None else 0.0,
        "BENIGN Precision" : r["0"]["precision"],
        "BENIGN Recall"    : r["0"]["recall"],
        "BENIGN F1"        : r["0"]["f1-score"],
        "ATTACK Precision" : r["1"]["precision"],
        "ATTACK Recall"    : r["1"]["recall"],
        "ATTACK F1"        : r["1"]["f1-score"],
        "cm"               : confusion_matrix(y_true, y_pred),
    }

def grade(v):
    if v >= 0.99: return "Excellent ✓"
    if v >= 0.95: return "Very Good ✓"
    if v >= 0.90: return "Good      ✓"
    if v >= 0.80: return "Fair"
    return "Needs Work ✗"

def print_block(title, m):
    W  = 57
    cm = m["cm"]
    tn, fp, fn, tp = cm.ravel()
    total = tn+fp+fn+tp
    print(f"\n  ┌{'─'*W}┐")
    print(f"  │  {title:<{W-2}}│")
    print(f"  ├{'─'*W}┤")
    print(f"  │  {'Metric':<28} {'Score':>8}   {'Grade':<16}│")
    print(f"  ├{'─'*W}┤")
    dividers = {"BENIGN Precision", "ATTACK Precision"}
    for k, v in m.items():
        if k == "cm": continue
        if k in dividers:
            print(f"  ├{'─'*W}┤")
        print(f"  │  {k:<28} {v:>8.4f}   {grade(v):<16}│")
    print(f"  ├{'─'*W}┤")
    print(f"  │  Confusion Matrix{' '*(W-18)}│")
    print(f"  │    TN: {tn:>6,} ({100*tn/total:.1f}%)   FP: {fp:>6,} ({100*fp/total:.1f}%){' '*8}│")
    print(f"  │    FN: {fn:>6,} ({100*fn/total:.1f}%)   TP: {tp:>6,} ({100*tp/total:.1f}%){' '*8}│")
    print(f"  └{'─'*W}┘")

# ── Header ────────────────────────────────────────────────────────────────────
W = 61
print("\n" + "═"*W)
print("  BEHAVIOURAL ANALYSIS OF NETWORK TRAFFIC")
print("  Final Results — Deep Learning Intrusion Detection")
print(f"  {datetime.now().strftime('%Y-%m-%d  %H:%M:%S')}")
print("═"*W)
print(f"\n  Dataset  : CICIDS 2017 (works with any network traffic dataset)")
print(f"  Models   : Supervised Autoencoder  +  BiLSTM with Attention")
print(f"  Test set : {len(X_test):,} | BENIGN: {(y_test==0).sum():,}  ATTACK: {(y_test==1).sum():,}")

all_m    = {}
ae_probs = None
l_probs  = None
y_seq    = None

# Autoencoder
ae_path = None
for ext in [".keras",".h5"]:
    p = os.path.join(MODEL_DIR, f"autoencoder_model{ext}")
    if os.path.exists(p): ae_path = p; break

if ae_path:
    print("\n  Computing Autoencoder predictions...")
    ae      = tf.keras.models.load_model(ae_path, compile=False)
    outputs = ae.predict(X_test, batch_size=1024, verbose=0)
    if isinstance(outputs, dict):   ae_probs = outputs["classification"].flatten()
    elif isinstance(outputs,(list,tuple)): ae_probs = next(o.flatten() for o in outputs if o.shape[-1]==1)
    else: ae_probs = outputs.flatten()
    m = get_metrics(y_test, (ae_probs>0.5).astype(int), ae_probs)
    all_m["Supervised Autoencoder"] = m
    print_block("MODEL 1 — Supervised Autoencoder", m)

# LSTM
lstm_path = None
for ext in [".keras",".h5"]:
    p = os.path.join(MODEL_DIR, f"bilstm_model{ext}")
    if os.path.exists(p): lstm_path = p; break

if lstm_path:
    print("\n  Computing BiLSTM predictions...")
    lstm    = tf.keras.models.load_model(
        lstm_path, compile=False)
    n_seq   = len(X_test) - TIMESTEPS
    shape   = (n_seq, TIMESTEPS, X_test.shape[1])
    strides = (X_test.strides[0], X_test.strides[0], X_test.strides[1])
    X_seq   = np.lib.stride_tricks.as_strided(
                  X_test, shape=shape, strides=strides).copy().astype(np.float32)
    y_seq = np.array([
        y_test[i + TIMESTEPS - 1]
        for i in range(n_seq)
    ], dtype=np.int32)
    l_probs = lstm.predict(X_seq, batch_size=1024, verbose=0).flatten()
    m = get_metrics(y_seq, (l_probs>0.5).astype(int), l_probs)
    all_m["BiLSTM + Attention"] = m
    print_block("MODEL 2 — BiLSTM with Attention (Early Detection)", m)

# Ensemble
if ae_probs is not None and l_probs is not None:
    ens_prob = 0.45 * ae_probs[TIMESTEPS:] + 0.55 * l_probs
    ens_pred = (ens_prob>0.5).astype(int)
    m = get_metrics(y_seq, ens_pred, ens_prob)
    all_m["Ensemble"] = m
    print_block("MODEL 3 — Weighted Ensemble (AE + BiLSTM)", m)

# Comparison table
if len(all_m) > 1:
    print(f"\n{'═'*W}\n  COMPARISON TABLE\n{'═'*W}")
    cw   = 16
    keys = ["Accuracy","Macro F1","ROC-AUC",
            "BENIGN Precision","ATTACK Precision",
            "BENIGN F1","ATTACK F1",
            "BENIGN Recall","ATTACK Recall"]
    print(f"  {'Metric':<22}" + "".join(f"{n:>{cw}}" for n in all_m))
    print("  "+"─"*(22+cw*len(all_m)))
    for k in keys:
        vals = [m[k] for m in all_m.values()]
        best = max(vals)
        row  = f"  {k:<22}"
        for v in vals:
            star = " ★" if abs(v-best)<1e-6 else "  "
            row += f"{v:>{cw-2}.4f}{star}"
        print(row)
    print("  ★ = best")

print(f"\n{'═'*W}")
print("  Both models are Deep Learning — no traditional ML used")
print("  Autoencoder: learns feature representations of attack behaviour")
print("  BiLSTM: detects temporal attack patterns BEFORE they complete")
print(f"{'═'*W}\n")
print("results.py completed successfully!")


═════════════════════════════════════════════════════════════
  BEHAVIOURAL ANALYSIS OF NETWORK TRAFFIC
  Final Results — Deep Learning Intrusion Detection
  2026-05-02  22:43:35
═════════════════════════════════════════════════════════════

  Dataset  : CICIDS 2017 (works with any network traffic dataset)
  Models   : Supervised Autoencoder  +  BiLSTM with Attention
  Test set : 66 | BENIGN: 0  ATTACK: 66

  Computing Autoencoder predictions...


C:\Users\Dell\anaconda3\Lib\site-packages\sklearn\metrics\_ranking.py:424: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(



  ┌─────────────────────────────────────────────────────────┐
  │  MODEL 1 — Supervised Autoencoder                       │
  ├─────────────────────────────────────────────────────────┤
  │  Metric                          Score   Grade           │
  ├─────────────────────────────────────────────────────────┤
  │  Accuracy                       0.2576   Needs Work ✗    │
  │  Macro F1                       0.2048   Needs Work ✗    │
  │  ROC-AUC                           nan   Needs Work ✗    │
  ├─────────────────────────────────────────────────────────┤
  │  BENIGN Precision               0.0000   Needs Work ✗    │
  │  BENIGN Recall                  0.0000   Needs Work ✗    │
  │  BENIGN F1                      0.0000   Needs Work ✗    │
  ├─────────────────────────────────────────────────────────┤
  │  ATTACK Precision               1.0000   Excellent ✓     │
  │  ATTACK Recall                  0.2576   Needs Work ✗    │
  │  ATTACK F1                      0.4096   Needs Work ✗   

C:\Users\Dell\anaconda3\Lib\site-packages\sklearn\metrics\_ranking.py:424: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
C:\Users\Dell\anaconda3\Lib\site-packages\sklearn\metrics\_ranking.py:424: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
